# 03 범주별 지표와 오차 분석

주 범주(A~E) · 날짜 유형 · 개소 유형 · 개소 · 교안 구분 · 상·하반기 · Chronos-2 공개일 전/후. 그리고 인력이 가장 많이 모자란 10건과 실수 4(합계로 평가).

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager

# 묶음 폴더(real_final 또는 final) 찾기 — notebooks/ 에서 열든, 묶음 폴더나 그 위(Colab 의 /content)에서 열든 된다
HERE = Path.cwd()
CANDIDATES = [p / sub for p in [HERE, *HERE.parents] for sub in ('data', 'real_final/data', 'final/data')]
DATA = next((c for c in CANDIDATES if (c / 'metrics.csv').exists()), None)
if DATA is None:
    raise FileNotFoundError(f'data/metrics.csv 를 찾지 못했습니다 (지금 위치: {HERE}). '
                            '묶음 폴더 전체(data · notebooks ...)를 받았는지, 노트북을 notebooks/ 에서 열었는지 확인하세요.')
BASE = DATA.parent                              # 묶음 폴더
# 01 노트북만 쓰는 src/ 와 원본 데이터: 묶음 폴더 안(real_final: src/ · raw/)이 먼저, 없으면 저장소 최상위
SRC = next((p for p in [BASE / 'src', BASE.parent / 'src'] if (p / 'hv.py').exists()), None)
RAW = next((p for p in [BASE / 'raw', BASE.parent] if (p / 'anchors.json').exists()), None)
if SRC is not None:
    sys.path.insert(0, str(SRC))
print('데이터:', DATA)

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 80)
HAVE = {f.name for f in font_manager.fontManager.ttflist}
KFONT = next((f for f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans KR'] if f in HAVE), None)
if KFONT is None:
    print('※ 한글 폰트가 없어 그림의 한글이 네모로 보일 수 있습니다 (Colab: !apt -qq install fonts-nanum 후 런타임 다시 시작)')
plt.rcParams.update({'font.family': KFONT or 'sans-serif', 'axes.unicode_minus': False, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
SITES = ['경복궁', '창덕궁', '창경궁', '덕수궁', '종묘', '동구릉', '광릉', '홍유릉', '사릉', '태릉', '정릉', '헌릉', '선릉', '의릉',
         '영휘원', '서오릉', '서삼릉', '융릉', '파주삼릉', '파주장릉', '김포장릉', '양주온릉', '세종대왕유적']   # 궁 → 종묘 → 왕릉
BLUE, ORANGE, AQUA, GRAY, VIOLET = '#2a78d6', '#eb6834', '#1baf7a', '#898781', '#6250d6'

M = pd.read_csv(DATA / 'metrics.csv')
ORDER = M.drop_duplicates('방식').sort_values('방식 순서')['방식'].tolist()

def table(axis, metric, conf='전날', models=None, scale=1):
    """축(axis)의 값별로 방식 × 값 표. scale=1e8 이면 억 원."""
    x = M[(M['축'] == axis) & (M['확정'] == conf)]
    t = x.pivot(index='방식', columns='값', values=metric) / scale
    idx = [m for m in ORDER if m in t.index and (models is None or m in models)]
    return t.reindex(idx)

KEY = ['1 직전 7일 평균', '2 + 서비스 수준 보정', '7 MLP + 마지막 수요일 (FVA 2차)', 'C3 Chronos-2 + 공휴일·무료 개방']

데이터: C:\Users\USER\Desktop\real_final\data


## 1. 주 범주 A~E (채점 축: 원점 주)

In [2]:
pd.concat({k: table('주범주', k, models=KEY, scale=1e8 if k == '비용' else 1) for k in ['WAPE', '적중률', '비용']}, axis=1).round(3)

WAPE                                                       적중률                                                        비용                                                
값                        A 명절 연휴 주 B 마지막 수요일 주 C 평일 공휴일 주(무료 없음) D 특별일 직후 주 E 평소 주 A 명절 연휴 주 B 마지막 수요일 주 C 평일 공휴일 주(무료 없음) D 특별일 직후 주 E 평소 주 A 명절 연휴 주 B 마지막 수요일 주 C 평일 공휴일 주(무료 없음) D 특별일 직후 주 E 평소 주
방식                                                                                                                                                                                                    
1 직전 7일 평균                   0.687       0.369             0.350      0.556  0.345     0.409       0.468             0.566      0.744  0.575     4.106       3.781             0.607      2.580  3.782
2 + 서비스 수준 보정                0.687       0.369             0.350      0.556  0.345     0.589       0.739             0.835      0.899  0.813     3.575       2.671             0.496      3.154  3.303
7 MLP + 마지막 수요일 (FVA 2차)     0.516       0.337             0.305      0.361  0.264     0.798       0.791             0.839      0.815  0.751     2.110       2.426             0.469      2.210  2.440
C3 Chronos-2 + 공휴일·무료 개방     0.441       0.315             0.289      0.301  0.218     0.779       0.731             0.820      0.834  0.823     1.955       2.318             0.437      1.860  1.891

## 2. 날짜 유형 (가이드 요구사항 7)

C3 의 약점: 마지막 수요일 적중률 0.49 — 무료 개방일에 모자라는 날이 절반.

In [3]:
DAY = ['명절 연휴(무료)', '마지막 수요일(무료)', '공휴일(무료 아님)', '주말', '평일']
pd.concat({k: table('날짜유형', k, models=KEY, scale=1e8 if k == '비용' else 1)[DAY] for k in ['WAPE', '적중률', '비용']}, axis=1).round(3)

WAPE                                            적중률                                             비용                                     
값                        명절 연휴(무료) 마지막 수요일(무료) 공휴일(무료 아님)     주말     평일 명절 연휴(무료) 마지막 수요일(무료) 공휴일(무료 아님)     주말     평일 명절 연휴(무료) 마지막 수요일(무료) 공휴일(무료 아님)     주말     평일
방식                                                                                                                                                                   
1 직전 7일 평균                   0.636       0.461      0.308  0.370  0.463     0.147       0.415      0.145  0.304  0.795     3.757       1.173      0.493  5.059  4.374
2 + 서비스 수준 보정                0.636       0.461      0.308  0.370  0.463     0.324       0.633      0.531  0.633  0.954     3.059       0.770      0.226  3.610  5.534
7 MLP + 마지막 수요일 (FVA 2차)     0.486       0.382      0.337  0.337  0.299     0.746       0.721      0.555  0.778  0.807     1.708       0.574      0.333  3.414  3.626
C3 Chronos-2 + 공휴일·무료 개방     0.459       0.384      0.363  0.273  0.253     0.689       0.485      0.488  0.810  0.843     1.699       0.605      0.359  2.789  3.008

## 3. 개소 유형 · 개소

In [4]:
pd.concat({k: table('개소유형', k, models=KEY, scale=1e8 if k == '비용' else 1) for k in ['WAPE', 'MASE', '적중률', '비용']}, axis=1).round(3)

WAPE                 MASE                  적중률                    비용              
값                             궁     왕릉     종묘      궁     왕릉     종묘      궁     왕릉     종묘       궁     왕릉     종묘
방식                                                                                                           
1 직전 7일 평균                0.408  0.510  0.874  1.281  1.136  2.577  0.534  0.592  0.594  10.964  2.618  1.275
2 + 서비스 수준 보정             0.408  0.510  0.874  1.281  1.136  2.577  0.785  0.809  0.796   9.634  2.364  1.202
7 MLP + 마지막 수요일 (FVA 2차)  0.321  0.380  0.523  0.992  0.860  1.523  0.767  0.792  0.738   7.161  1.664  0.830
C3 Chronos-2 + 공휴일·무료 개방  0.274  0.344  0.465  0.835  0.790  1.346  0.794  0.809  0.760   6.214  1.493  0.753

In [5]:
st = table('개소', '비용', models=['1 직전 7일 평균', 'C3 Chronos-2 + 공휴일·무료 개방'], scale=1e6).T
st['절감'] = st.iloc[:, 0] - st.iloc[:, 1]; st['절감률'] = st['절감'] / st.iloc[:, 0]
st.sort_values('절감', ascending=False).round(2)

방식,1 직전 7일 평균,C3 Chronos-2 + 공휴일·무료 개방,절감,절감률
값,,,,
경복궁,471.95,257.54,214.41,0.45
창덕궁,201.92,106.30,95.62,0.47
창경궁,193.25,105.49,87.76,0.45
덕수궁,229.24,152.09,77.15,0.34
종묘,127.49,75.33,52.16,0.41
융릉,43.08,20.54,22.54,0.52
서오릉,33.49,15.01,18.48,0.55
세종대왕유적,33.89,20.16,13.73,0.41
선릉,37.34,26.30,11.05,0.30


## 4. 교안 구분 · 상·하반기 · Chronos-2 공개일(2025-10-17) 전/후

공개일 전 원점은 사전학습 자료와 겹쳤을 수 있다. 공개일 후 원점에서도 C3 가 이기는지 본다.

In [6]:
for ax_ in ['교안 구분', '반기', '공개일']:
    print('■', ax_); display(pd.concat({k: table(ax_, k, models=KEY, scale=1e8 if k == '비용' else 1) for k in ['WAPE', 'MASE', '비용']}, axis=1).round(3))

■ 교안 구분


WAPE              MASE                비용         
값                        공휴일 없는 주 공휴일 있는 주 공휴일 없는 주 공휴일 있는 주 공휴일 없는 주 공휴일 있는 주
방식                                                                            
1 직전 7일 평균                  0.413    0.525    1.089    1.725    8.747    6.110
2 + 서비스 수준 보정               0.413    0.525    1.089    1.725    8.217    4.983
7 MLP + 마지막 수요일 (FVA 2차)    0.309    0.418    0.738    1.559    6.249    3.406
C3 Chronos-2 + 공휴일·무료 개방    0.262    0.376    0.671    1.386    5.230    3.231

■ 반기


WAPE          MASE            비용       
값                        상반기 원점 하반기 원점 상반기 원점 하반기 원점 상반기 원점 하반기 원점
방식                                                                
1 직전 7일 평균                0.410  0.474  1.136  1.312  6.516  8.341
2 + 서비스 수준 보정             0.410  0.474  1.136  1.312  5.580  7.620
7 MLP + 마지막 수요일 (FVA 2차)  0.337  0.341  0.901  0.923  4.188  5.467
C3 Chronos-2 + 공휴일·무료 개방  0.284  0.301  0.802  0.843  3.560  4.901

■ 공개일


WAPE          MASE             비용       
값                         공개일 전  공개일 후  공개일 전  공개일 후   공개일 전  공개일 후
방식                                                                 
1 직전 7일 평균                0.422  0.507  1.134  1.560  11.163  3.694
2 + 서비스 수준 보정             0.422  0.507  1.134  1.560   9.608  3.591
7 MLP + 마지막 수요일 (FVA 2차)  0.346  0.320  0.913  0.908   7.008  2.647
C3 Chronos-2 + 공휴일·무료 개방  0.299  0.276  0.822  0.823   6.227  2.234

## 5. 인력이 가장 많이 모자란 10건 (C3)

원인 셋: 명절 연휴 후반 7건(가을 궁중문화축전과 겹침) · 규칙 밖 무료 개방 2건(10-29~11-09) · 행사일 1건(종묘대제).

In [7]:
T10 = pd.read_csv(DATA / 'top10_shortage.csv')
display(T10[['순위', 'date', 'weekday', 'site', 'y', 'q80', 'short', 'cost', '원인 분류', '원인']].round(1))
print('10건 비용 합', round(T10['cost'].sum() / 1e8, 3), '억')
T10.groupby('원인 분류')['cost'].agg(['size', 'sum']).assign(sum=lambda d: (d['sum'] / 1e8).round(3))

,순위,date,weekday,site,y,q80,short,cost,원인 분류,원인
0,1,2025-10-08,수,경복궁,126362.0,74702.3,25.8,15497916.6,명절 연휴 후반,"추석 연휴 끝(당일 +2일, 대체 휴일) — 「명절 연휴 0/1」로는 연휴 안의 위..."
1,2,2025-10-09,목,경복궁,98771.0,52994.3,22.9,13733017.8,명절 연휴 후반,추석 연휴 마지막 날(한글날) + 궁중문화축전 둘째 날
2,3,2025-05-04,일,종묘,44337.0,5315.8,19.5,11706360.4,행사일,종묘대제(5월 첫째 일요일) — 정전 보수(2020~2025-04) 뒤 6년 만에 ...
3,4,2025-10-08,수,창경궁,58122.0,25593.5,16.3,9758552.8,명절 연휴 후반,"추석 연휴 끝(당일 +2일, 대체 휴일) — 「명절 연휴 0/1」로는 연휴 안의 위..."
4,5,2025-11-08,토,덕수궁,66257.0,36451.1,14.9,8941763.3,규칙 밖 무료 개방,2025-10-29~11-09 4대 궁·종묘·조선왕릉·세종대왕유적 무료 개방(코리아...
5,6,2025-10-09,목,창경궁,44822.0,17361.6,13.7,8238121.5,명절 연휴 후반,추석 연휴 마지막 날(한글날) + 궁중문화축전 둘째 날
6,7,2025-10-08,수,창덕궁,56508.0,29118.1,13.7,8216963.9,명절 연휴 후반,"추석 연휴 끝(당일 +2일, 대체 휴일) — 「명절 연휴 0/1」로는 연휴 안의 위..."
7,8,2025-10-08,수,덕수궁,49985.0,23406.1,13.3,7973671.9,명절 연휴 후반,"추석 연휴 끝(당일 +2일, 대체 휴일) — 「명절 연휴 0/1」로는 연휴 안의 위..."
8,9,2025-11-04,화,창덕궁,45452.0,19083.1,13.2,7910658.7,규칙 밖 무료 개방,2025-10-29~11-09 4대 궁·종묘·조선왕릉·세종대왕유적 무료 개방(코리아...
9,10,2025-10-09,목,창덕궁,44893.0,19489.9,12.7,7620917.2,명절 연휴 후반,추석 연휴 마지막 날(한글날) + 궁중문화축전 둘째 날


10건 비용 합 0.996 억


,size,sum
원인 분류,,
규칙 밖 무료 개방,2,0.169
명절 연휴 후반,7,0.710
행사일,1,0.117


## 6. 자주 하는 실수 4 — 합계로 평가하면

23개소를 날짜별로 합쳐 채점하면 개소 간 +·− 오차가 상쇄돼 좋아 보인다.

In [8]:
BT = pd.read_parquet(DATA / 'backtests.parquet', filters=[('seed', '==', 0), ('확정', '==', '전날')])
BT = BT[BT['방식'].isin(['1 직전 7일 평균', 'C3 Chronos-2 + 공휴일·무료 개방']) & (BT['status'] == 'open')]
def wape(d): return (d['y'] - d['yhat']).abs().sum() / d['y'].sum()
out = {}
for m, d in BT.groupby('방식'):
    tot = d.groupby('date')[['y', 'yhat']].sum()
    grp = d.groupby(['date', 'group'])[['y', 'yhat']].sum()
    out[m] = {'개소×날짜 (우리 방식)': wape(d), '개소 유형 합계×날짜': wape(grp), '23개소 합계×날짜': wape(tot)}
pd.DataFrame(out).T.round(3)

,개소×날짜 (우리 방식),개소 유형 합계×날짜,23개소 합계×날짜
1 직전 7일 평균,0.444,0.416,0.399
C3 Chronos-2 + 공휴일·무료 개방,0.293,0.261,0.250
